# Experiment 12: Neural Machine Translation (NMT) System using Pretrained Transformers & BLEU Evaluation

### Problem Statement:
Develop a Machine Translation system using pre-trained transformer architectures to translate text between two languages and evaluate translation quality.


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import pandas as pd
import numpy as np

print("Translation architectures and evaluation metrics imported.")


/Users/jeet/miniconda3/envs/ml_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Translation architectures and evaluation metrics imported.


In [2]:
model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.eval()

# Model Architecture Parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model Name            : {model_name}")
print(f"Total Parameters      : {total_params:,}")
print(f"Source Vocab Size     : {tokenizer.vocab_size:,}")
print(f"Encoder Layers        : {len(model.model.encoder.layers)}")
print(f"Decoder Layers        : {len(model.model.decoder.layers)}")
print(f"Hidden Dimension (d_k): {model.config.d_model}")


Loading weights:   0%|                                 | 0/258 [00:00<?, ?it/s]

Loading weights: 100%|████████████████████| 258/258 [00:00<00:00, 59242.88it/s]

Model Name            : Helsinki-NLP/opus-mt-en-fr
Total Parameters      : 75,133,952
Source Vocab Size     : 59,514
Encoder Layers        : 6
Decoder Layers        : 6
Hidden Dimension (d_k): 512


In [3]:
test_sentences = [
    "Artificial intelligence is transforming modern science and education.",
    "The researchers proposed a novel neural network architecture for natural language understanding.",
    "She lives in a beautiful house near the river in the south of France.",
    "Time flies like an arrow, but fruit flies like a banana." # Idiomatic / Polysemous
]

def translate_text(text, model, tokenizer, num_beams=4, max_length=64):
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            num_beams=num_beams,
            max_length=max_length,
            early_stopping=True
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

decoding_comparison = []
for sent in test_sentences:
    greedy_trans = translate_text(sent, model, tokenizer, num_beams=1)
    beam_trans = translate_text(sent, model, tokenizer, num_beams=4)
    decoding_comparison.append({
        "English Source": sent,
        "Greedy Translation (beam=1)": greedy_trans,
        "Beam Search Translation (beam=4)": beam_trans
    })

df_trans = pd.DataFrame(decoding_comparison)
print("--- Translation Comparison: Greedy vs Beam Search ---")
for idx, row in df_trans.iterrows():
    print(f"Source : '{row['English Source']}'")
    print(f"Greedy : '{row['Greedy Translation (beam=1)']}'")
    print(f"Beam   : '{row['Beam Search Translation (beam=4)']}'\n")


[transformers] The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


--- Translation Comparison: Greedy vs Beam Search ---
Source : 'Artificial intelligence is transforming modern science and education.'
Greedy : 'L'intelligence artificielle transforme la science et l'éducation modernes.'
Beam   : 'L'intelligence artificielle transforme la science et l'éducation modernes.'

Source : 'The researchers proposed a novel neural network architecture for natural language understanding.'
Greedy : 'Les chercheurs ont proposé une architecture de réseau neuronal nouvelle pour la compréhension du langage naturel.'
Beam   : 'Les chercheurs ont proposé une nouvelle architecture de réseau neuronal pour la compréhension du langage naturel.'

Source : 'She lives in a beautiful house near the river in the south of France.'
Greedy : 'Elle vit dans une belle maison près de la rivière dans le sud de la France.'
Beam   : 'Elle vit dans une belle maison près de la rivière dans le sud de la France.'

Source : 'Time flies like an arrow, but fruit flies like a banana.'
Greedy : 

In [4]:
eval_pairs = [
    {
        "source": "Natural Language Processing allows computers to understand human language.",
        "reference": ["Le traitement du langage naturel permet aux ordinateurs de comprendre le langage humain."],
        "human_ref_str": "Le traitement du langage naturel permet aux ordinateurs de comprendre le langage humain."
    },
    {
        "source": "Deep learning models require massive datasets and high performance computers.",
        "reference": ["Les modèles d'apprentissage profond nécessitent des ensembles de données massifs et des ordinateurs performants."],
        "human_ref_str": "Les modèles d'apprentissage profond nécessitent des ensembles de données massifs et des ordinateurs performants."
    },
    {
        "source": "We traveled to Paris by train yesterday morning.",
        "reference": ["Nous avons voyagé à Paris en train hier matin."],
        "human_ref_str": "Nous avons voyagé à Paris en train hier matin."
    },
    {
        "source": "The weather today is exceptionally sunny and warm.",
        "reference": ["Le temps aujourd'hui est exceptionnellement ensoleillé et chaud."],
        "human_ref_str": "Le temps aujourd'hui est exceptionnellement ensoleillé et chaud."
    }
]

smooth_fn = SmoothingFunction().method1

bleu_records = []
for item in eval_pairs:
    src = item["source"]
    ref_tokens = [item["reference"][0].lower().split()]
    
    # Generate Translation
    cand_str = translate_text(src, model, tokenizer, num_beams=4)
    cand_tokens = cand_str.lower().split()
    
    # Compute Individual N-gram Precisions & Cumulative BLEU
    b1 = sentence_bleu(ref_tokens, cand_tokens, weights=(1, 0, 0, 0), smoothing_function=smooth_fn)
    b2 = sentence_bleu(ref_tokens, cand_tokens, weights=(0.5, 0.5, 0, 0), smoothing_function=smooth_fn)
    b3 = sentence_bleu(ref_tokens, cand_tokens, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smooth_fn)
    b4 = sentence_bleu(ref_tokens, cand_tokens, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth_fn)
    
    bleu_records.append({
        "Source (EN)": src,
        "Candidate (MT)": cand_str,
        "Reference (FR)": item["human_ref_str"],
        "BLEU-1": round(b1, 3),
        "BLEU-2": round(b2, 3),
        "BLEU-3": round(b3, 3),
        "BLEU-4": round(b4, 3)
    })

df_bleu = pd.DataFrame(bleu_records)
print("--- Quantitative BLEU Score Evaluation Report ---")
for idx, r in df_bleu.iterrows():
    print(f"[{idx+1}] Source   : {r['Source (EN)']}")
    print(f"    Candidate: {r['Candidate (MT)']}")
    print(f"    Reference: {r['Reference (FR)']}")
    print(f"    Scores   : BLEU-1: {r['BLEU-1']} | BLEU-2: {r['BLEU-2']} | BLEU-3: {r['BLEU-3']} | BLEU-4: {r['BLEU-4']}\n")


--- Quantitative BLEU Score Evaluation Report ---
[1] Source   : Natural Language Processing allows computers to understand human language.
    Candidate: Le traitement du langage naturel permet aux ordinateurs de comprendre le langage humain.
    Reference: Le traitement du langage naturel permet aux ordinateurs de comprendre le langage humain.
    Scores   : BLEU-1: 1.0 | BLEU-2: 1.0 | BLEU-3: 1.0 | BLEU-4: 1.0

[2] Source   : Deep learning models require massive datasets and high performance computers.
    Candidate: Les modèles d'apprentissage profond nécessitent des ensembles de données massifs et des ordinateurs de haute performance.
    Reference: Les modèles d'apprentissage profond nécessitent des ensembles de données massifs et des ordinateurs performants.
    Scores   : BLEU-1: 0.812 | BLEU-2: 0.806 | BLEU-3: 0.801 | BLEU-4: 0.792

[3] Source   : We traveled to Paris by train yesterday morning.
    Candidate: Nous avons voyagé à Paris en train hier matin.
    Reference: Nous 

In [5]:
nuance_sentences = [
    "The beautiful white house and the beautiful white car.", # Tests gender: 'belle maison blanche' (fem) vs 'belle voiture blanche' (fem)
    "He speaks to the doctor and to the students.",          # Tests contractions: 'au médecin' vs 'aux étudiants'
    "It is important that you be present tomorrow."          # Tests subjunctive mood
]

print("--- Qualitative Linguistic Agreement Tests ---")
for sent in nuance_sentences:
    trans = translate_text(sent, model, tokenizer, num_beams=4)
    print(f"Source English : '{sent}'")
    print(f"French Target  : '{trans}'\n")


--- Qualitative Linguistic Agreement Tests ---
Source English : 'The beautiful white house and the beautiful white car.'
French Target  : 'La belle maison blanche et la belle voiture blanche.'



Source English : 'He speaks to the doctor and to the students.'
French Target  : 'Il parle au médecin et aux étudiants.'

Source English : 'It is important that you be present tomorrow.'
French Target  : 'Il est important que vous soyez présent demain.'



In [6]:
def interactive_translate_and_evaluate(english_text, reference_french=None):
    french_translation = translate_text(english_text, model, tokenizer, num_beams=4)
    
    print("=================================================================")
    print(f"INPUT (EN)   : '{english_text}'")
    print(f"OUTPUT (FR)  : '{french_translation}'")
    
    if reference_french:
        ref_tokens = [reference_french.lower().split()]
        cand_tokens = french_translation.lower().split()
        score = sentence_bleu(ref_tokens, cand_tokens, smoothing_function=smooth_fn)
        print(f"REFERENCE    : '{reference_french}'")
        print(f"BLEU SCORE   : {score*100:.2f} / 100.0")
    print("=================================================================\n")

interactive_translate_and_evaluate(
    "Machine translation powered by deep learning is remarkably accurate.",
    "La traduction automatique alimentée par l'apprentissage profond est remarquablement précise."
)


INPUT (EN)   : 'Machine translation powered by deep learning is remarkably accurate.'
OUTPUT (FR)  : 'La traduction automatique alimentée par l'apprentissage profond est remarquablement précise.'
REFERENCE    : 'La traduction automatique alimentée par l'apprentissage profond est remarquablement précise.'
BLEU SCORE   : 100.00 / 100.0

